In [ ]:
## 1. Mount Google Drive (اختیاری)

In [ ]:
from google.colab import drive

MODE = "MOUNT"  # "MOUNT" یا "UNMOUNT"

drive.mount._DEBUG = False

if MODE == "MOUNT":
    drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass
    get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

In [ ]:
## 2. Setup And Update ComfyUI

In [ ]:
import os

DRIVE_PATH = ""
UPDATE_COMFY_UI = True

WORKSPACE = '/content/ComfyUI'
if DRIVE_PATH:
    WORKSPACE = f"{DRIVE_PATH}/ComfyUI"

os.makedirs(os.path.dirname(WORKSPACE) or "/content", exist_ok=True)

if not os.path.isdir(WORKSPACE):
    print("-= Initial setup ComfyUI =-")
    %cd /content
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print(f"ComfyUI already exists at {WORKSPACE}")

%cd {WORKSPACE}

if UPDATE_COMFY_UI:
    print("-= Updating ComfyUI =-")
    !git pull

# نصب torch با CUDA
print("-= Install PyTorch with CUDA =-")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# نصب بقیه dependencies
print("-= Install other dependencies =-")
!pip install -q xformers!=0.0.18 -r requirements.txt

# تأیید CUDA
import torch
assert torch.cuda.is_available(), "❌ CUDA not available! Runtime → Change runtime type → T4 GPU"
print(f"✅ CUDA OK: {torch.cuda.get_device_name(0)}")

assert os.path.isfile(f"{WORKSPACE}/main.py"), f"main.py not found!"
print(f"✅ ComfyUI ready at {WORKSPACE}")

In [ ]:
%cd {WORKSPACE}

# دانلود Text Encoder
!wget -c https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -P ./models/text_encoders/

# دانلود Diffusion Model
!wget -c https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors -P ./models/diffusion_models/

In [ ]:
## 3. Models Download

In [ ]:
%cd {WORKSPACE}
!apt-get -y install -qq aria2

import os
from google.colab import userdata

# توکن اختیاری Civitai
try:
    CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except Exception:
    CIVITAI_API_TOKEN = None

if CIVITAI_API_TOKEN:
    print("Loaded API key: ✅")
else:
    print("Loaded API key: ⚠️  Not found — only public downloads will work")

os.makedirs("./models/checkpoints", exist_ok=True)
os.makedirs("./models/vae", exist_ok=True)


def install_custom_node(url):
    %cd {WORKSPACE}/custom_nodes
    !git clone {url}
    %cd {WORKSPACE}


def downloadModel(url, filename=None):
    if 'huggingface.co' in url:
        if filename is None:
            filename = url.split('/')[-1].removesuffix('?download=true')
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -o "{filename}"
    else:
        # civitai
        if CIVITAI_API_TOKEN:
            sep = "&" if "?" in url else "?"
            full = f"{url}{sep}token={CIVITAI_API_TOKEN}"
        else:
            full = url
        if filename:
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full}" -o "{filename}"
        else:
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full}"


# --- Custom nodes (اختیاری) ---
# install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
# install_custom_node('https://github.com/ltdrdata/ComfyUI-Impact-Pack')


# --- دانلود مدل‌ها ---
%cd {WORKSPACE}/models/checkpoints

# CyberRealistic Pony
downloadModel('https://civitai.com/api/download/models/2071650')

# مدل‌های دیگه (uncomment کن اگه می‌خوای)
# downloadModel('https://civitai.com/api/download/models/201514')  # Protovision XL
# downloadModel('https://civitai.com/api/download/models/471120')  # Juggernaut XL Hyper
# downloadModel('https://civitai.com/api/download/models/456194')  # Juggernaut X
# downloadModel('https://civitai.com/api/download/models/288982')  # Juggernaut v8

# Huggingface example
# downloadModel('https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors?download=true')

%cd {WORKSPACE}

In [ ]:
## 4. START ComfyUI (Cloudflare Tunnel)

In [ ]:
# نصب پلاگین Direct Model Downloader
!git clone https://github.com/hradec/comfyui-direct-model-downloader.git /content/ComfyUI/custom_nodes/comfyui-direct-model-downloader

In [ ]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# نصب cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, threading, time, socket, os

PORT = 8188
MAIN = f"{WORKSPACE}/main.py"
assert os.path.isfile(MAIN), "main.py not found!"

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break
    print("\nComfyUI finished loading, launching cloudflared...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(PORT,)).start()
%cd {WORKSPACE}
!python main.py --dont-print-server